In [1]:
from datasets import load_dataset

ds = load_dataset("Jzuluaga/uwb_atcc", split="test", streaming=True)
ds

IterableDataset({
    features: ['id', 'audio', 'text', 'segment_start_time', 'segment_end_time', 'duration'],
    num_shards: 1
})

In [2]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "jlvdoorn/whisper-large-v3-atco2-asr"

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, dtype=torch.float16).to("mps")
model.eval()
print("loaded on", model.device)

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

loaded on mps:0


In [2]:
ex = next(iter(ds))

print(ex["id"])
print(ex["text"])
print(ex["segment_start_time"], ex["segment_end_time"], ex["duration"])

uwb-atcc_TWR-34720N_001179_001375_PI
for clearance we have information lima
11.789999961853027 13.75 1.9600000381469727


In [3]:
audio = ex["audio"]
samples = audio.get_all_samples()

print("sample rate:", samples.sample_rate)
print("shape:", samples.data.shape)
print("seconds:", samples.data.shape[-1] / samples.sample_rate)

sample rate: 16000
shape: torch.Size([1, 31360])
seconds: 1.96


In [4]:
from IPython.display import Audio
Audio(samples.data.numpy(), rate=samples.sample_rate)

In [5]:
for row in ds.select_columns(["id", "text"]).take(20):
    print(row["id"][-30:], "|", row["text"])

cc_TWR-34720N_001179_001375_PI | for clearance we have information lima
cc_TWR-34720N_001666_001952_AT | el al five two two startup approved time four five
cc_TWR-34720N_002001_002559_AT | clearance to telaviv via voz one hotel departure squawk alfa
cc_TWR-34720N_002594_002669_AT | one four four
cc_TWR-34720N_003268_003531_PI | cleared via voz one hotel
cc_TWR-34720N_003568_003790_PI | departure runway one three
cc_TWR-34720N_003839_004090_PI | one four four two el al five
cc_TWR-34720N_004252_004326_AT | correct
cc_TWR-a1WcrN_000157_000469_AT | nor shuttle one five one five line up runway three one
cc_TWR-a3o8f0_000000_000352_AT | opera jet three zero zero just for confirmation one hotel
cc_TWR-a3o8f0_000409_000519_PI | one hotel opera
cc_TWR-A5lZHJ_000000_000613_AT | csa three seven alfa runway three one cleared for takeoff wind one six zero degrees five knots
cc_TWR-A5lZHJ_000649_000787_PI | runway three one cleared for takeoff
cc_TWR-A5lZHJ_000824_000943_PI | en three alfa
cc_TWR-A

<h3> I have now downloaded the full whisper model after streaming the clips</h3>

<h5>spike 2</h5>

In [3]:
clips = list(ds.take(3))
[c["id"][-30:] for c in clips]

['cc_TWR-34720N_001179_001375_PI',
 'cc_TWR-34720N_001666_001952_AT',
 'cc_TWR-34720N_002001_002559_AT']

In [5]:
ex = clips[0]
features = processor(ex["audio"]["array"], sampling_rate=16000, return_tensors="pt").input_features
features = features.to("mps", torch.float16)

out = model.generate(
    features,
    language="en", task="transcribe",
    use_cache=False,
    num_beams=1, do_sample=False,
    return_dict_in_generate=True, output_scores=True,
)

print("REF:", ex["text"])
print("HYP:", processor.batch_decode(out.sequences, skip_special_tokens=True)[0])

REF: for clearance we have information lima
HYP:  we have information lima 


In [6]:
for step in out.scores:
    p, tok = step[0].float().softmax(-1).max(-1)
    print(f"{processor.decode([tok.item()])!r:>14}  {p.item():.3f}")

         ' we'  0.206
       ' have'  1.000
' information'  0.517
        ' lim'  0.997
           'a'  1.000
           ' '  0.999
'<|endoftext|>'  1.000


<h6>i am now going to call all 3 clips at once and see the prob score again, here i have noticed the model lacked confidence with some words</h6>

In [7]:
import time

def transcribe(array):
    f = processor(array, sampling_rate=16000, return_tensors="pt").input_features.to("mps", torch.float16)
    o = model.generate(f, language="en", task="transcribe", num_beams=1, do_sample=False,
                       use_cache=False, return_dict_in_generate=True, output_scores=True)
    text = processor.batch_decode(o.sequences, skip_special_tokens=True)[0]
    tokens = []
    for step in o.scores:
        p, tok = step[0].float().softmax(-1).max(-1)
        tokens.append((processor.decode([tok.item()]), round(p.item(), 3)))
    return text, tokens

for c in clips:
    start = time.perf_counter()
    text, tokens = transcribe(c["audio"]["array"])
    secs = time.perf_counter() - start
    print("REF:", c["text"])
    print("HYP:", text)
    print("least sure:", min(tokens, key=lambda t: t[1]), f"| {secs:.1f} s")
    print()

REF: for clearance we have information lima
HYP:  we have information lima 
least sure: (' we', 0.206) | 5.7 s

REF: el al five two two startup approved time four five
HYP: hello Five Two Two startup is approved time four five 
least sure: ('he', 0.515) | 7.6 s

REF: clearance to telaviv via voz one hotel departure squawk alfa
HYP:  clearance to Tel Aviv via vožice one hotel departure squawk alpha 
least sure: (' vo', 0.283) | 8.4 s



<h5>spike 3</h5>

In [10]:
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")
client = genai.Client()

for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [11]:
# extraction
from typing import Literal
from pydantic import BaseModel

class Item(BaseModel):
    type: Literal["level", "heading", "speed", "frequency", "squawk", "qnh", "runway"]
    value: str
    unit: str | None = None
    qualifier: str | None = None

class Clearance(BaseModel):
    callsign: str | None
    items: list[Item]

In [13]:
from google.genai import types

client = genai.Client(http_options=types.HttpOptions(timeout=60_000))

r = client.models.generate_content(model=MODEL, contents="Say hi in one word.")
print(r.text)

Hello.


In [16]:
# only one extraction
MODEL = "gemini-3.5-flash-lite"

instruction = "lufthansa four five two descend flight level one two zero turn left heading two seven zero"

prompt = f"""Extract the air traffic control clearance from this radio transmission.
Write numbers as digits. Use null for the callsign if none is spoken.

Transmission: {instruction}"""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": Clearance.model_json_schema(),
        "temperature": 0,
        "thinking_config": {"thinking_level": "low"}
    },
)
print(resp.text)

{"callsign": "Lufthansa 452", "items": [{"type": "level", "value": "120", "unit": "flight level", "qualifier": null}, {"type": "heading", "value": "270", "unit": "degrees", "qualifier": "left"}]}


In [17]:
# i am now going to validate it 
atco = Clearance.model_validate_json(resp.text)
atco

Clearance(callsign='Lufthansa 452', items=[Item(type='level', value='120', unit='flight level', qualifier=None), Item(type='heading', value='270', unit='degrees', qualifier='left')])

In [18]:
readback = "descend flight level one three zero left heading two seven zero lufthansa four five two"

resp2 = client.models.generate_content(
    model=MODEL,
    contents=prompt.replace(instruction, readback),
    config={"response_mime_type": "application/json",
            "response_json_schema": Clearance.model_json_schema(),
            "temperature": 0},
)
pilot = Clearance.model_validate_json(resp2.text)
print(atco)
print(pilot)

callsign='Lufthansa 452' items=[Item(type='level', value='120', unit='flight level', qualifier=None), Item(type='heading', value='270', unit='degrees', qualifier='left')]
callsign='lufthansa four five two' items=[Item(type='level', value='130', unit='flight level', qualifier='descend'), Item(type='heading', value='270', unit=None, qualifier='left')]
